# E0 · Exportar, quantizar e medir o twitter-XLM-R para servir

Parte dos pesos que o notebook 1 deixou em `Meu Drive/Luciola_treino_v5_gpu/modelos/`.
Responde as três perguntas que decidem se ele pode substituir o stack na
Hetzner:

1. **Cabe?** Tamanho em disco de cada variante.
2. **Responde a tempo?** Latência em **CPU**, que é o que a Hetzner tem, e com
   o mesmo teto de 2 threads da máquina. O stack servido hoje faz 1,5 ms.
3. **Passa no gate de viés?** A sonda de identidade é condição de release
   declarada em `methodology/pt_recall_x_vies.md`. Quantização muda score, então
   a sonda roda em cada variante, não só no original.

**Runtime: CPU.** Troque em Ambiente de execução, Alterar tipo. Rodar em GPU aqui
mede a máquina errada.

Leva uns 20 a 40 minutos.

## 1 · Confirmar que o runtime é CPU

In [ ]:
import torch
assert not torch.cuda.is_available(), (
    'Este notebook mede latencia de CPU, que e o que a Hetzner tem. '
    'Troque o runtime para CPU e rode de novo.')
!cat /proc/cpuinfo | grep 'model name' | head -2
!nproc

## 2 · Dependências, repositório e imports (célula única)

Sem `optimum` de propósito. Ele arrasta `diffusers` na cadeia de export e força
o `huggingface-hub` para trás, o que quebra os pacotes que o Colab já traz.
`torch.onnx` e `onnxruntime.quantization` fazem o mesmo trabalho com duas
dependências em vez de dez, e deixam o grafo exportado à vista.

In [ ]:
!pip install -q --upgrade-strategy only-if-needed 'onnx>=1.16' 'onnxruntime>=1.18' \
    'scikit-learn>=1.4' 'pyarrow>=15' 'pyyaml>=6.0' 'lingua-language-detector>=2.0'

!git clone -q https://github.com/isasaade-23/hate-speech-nlp-en-pt.git /content/repo

import json, os, shutil, sys, time, zipfile
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
from onnxruntime.quantization import QuantType, quantize_dynamic
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

REPO = Path('/content/repo')
sys.path.insert(0, str(REPO / 'src'))
sys.path.insert(0, str(REPO / 'scripts'))
os.chdir(REPO)

# as frases da sonda vem do repositorio: nao podem divergir das do gate local
from pt_identity_probe import CONTROLES, FRASES
print('onnxruntime', ort.__version__, '| torch', torch.__version__)
print(f'sonda: {len(FRASES)} frases neutras, {len(CONTROLES)} controles de odio')

## 3 · Os pesos e o corpus, do Drive

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/Luciola_treino_v5_gpu'
from google.colab import drive
drive.mount('/content/drive')

MODELOS = Path(DRIVE_DIR) / 'modelos'
CARTAO = MODELOS / 'twitter_xlmr_strict_s42' / 'luciola_serve.json'
if not CARTAO.exists():
    achados = sorted(q.name for q in MODELOS.iterdir()) if MODELOS.exists() else []
    print('Faltam os pesos. Este notebook parte do que o notebook 1 grava no Drive.')
    print('  esperado:', CARTAO)
    print('  encontrado em', MODELOS, ':', achados or '(a pasta nao existe)')
    print('Rode colab_xlmr_treino.ipynb ate o fim, com runtime GPU.')
    print('Ele so escreve o luciola_serve.json na ultima celula de treino, entao')
    print('sessao derrubada no meio nao deixa nada aproveitavel.')
    raise SystemExit('pesos ausentes')

META = json.load(open(CARTAO))
PESOS = Path(DRIVE_DIR) / 'modelos' / META['model_id']
MAXLEN, TEXT_COL = META['max_length'], META['text_column']
print('pesos:', PESOS, '| limiar do treino:', META['threshold'])

CORPUS = pd.read_parquet(Path(DRIVE_DIR) / 'corpus_strict.parquet')
VA = CORPUS[(CORPUS.split == 'val') & CORPUS.language.isin(['en', 'pt'])].reset_index(drop=True)
TE = CORPUS[(CORPUS.split == 'test') & CORPUS.language.isin(['en', 'pt'])].reset_index(drop=True)
print(f'val={len(VA)} test={len(TE)}')

## 4 · Exportar para ONNX e quantizar em int8

Quantização **dinâmica** de peso: não exige conjunto de calibração e o erro fica
pequeno em modelo de sequência curta. Eixos dinâmicos em lote e comprimento,
para o mesmo grafo servir texto avulso e lote de documento. `per_channel=True`
porque uma escala por canal de saída erra menos que uma escala para a matriz
inteira, e custa pouco.

A CPU do servidor é **AMD EPYC Genoa** e traz `avx512_vnni`, verificado em
11/09. Na quantização dinâmica o onnxruntime escolhe o caminho VNNI em tempo de
execução pela CPU que encontra, então não há nada a fixar aqui: o mesmo arquivo
roda rápido lá e continua correto em máquina sem VNNI.

In [ ]:
tok = AutoTokenizer.from_pretrained(PESOS)
torch_model = AutoModelForSequenceClassification.from_pretrained(PESOS).eval()

ONNX_DIR = Path('/content/onnx'); ONNX_DIR.mkdir(exist_ok=True)
FP32, INT8 = ONNX_DIR / 'model_fp32.onnx', ONNX_DIR / 'model_int8.onnx'

ex = tok(['exemplo curto para tracar o grafo'], return_tensors='pt',
         truncation=True, max_length=MAXLEN, padding='max_length')
with torch.no_grad():
    torch.onnx.export(
        torch_model, (ex['input_ids'], ex['attention_mask']), str(FP32),
        input_names=['input_ids', 'attention_mask'], output_names=['logits'],
        dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'},
                      'attention_mask': {0: 'batch', 1: 'seq'},
                      'logits': {0: 'batch'}},
        opset_version=17, do_constant_folding=True)
print('fp32 exportado')

quantize_dynamic(FP32, INT8, weight_type=QuantType.QInt8, per_channel=True)
print('int8 pronto')

def mb(p):
    p = Path(p)
    tot = sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) if p.is_dir() else p.stat().st_size
    return round(tot / 1e6, 1)

TAM = {'torch_fp32': mb(PESOS), 'onnx_fp32': mb(FP32), 'onnx_int8': mb(INT8)}
for k, v in TAM.items():
    print(f'{k:12s} {v:8.1f} MB')

## 5 · As três variantes atrás da mesma interface

As sessões ONNX rodam com **2 threads**, que é o número de vCPU da Hetzner.
Sem esse teto o Colab mediria uma máquina que não existe do outro lado.

In [ ]:
THREADS = 2

def sessao(caminho):
    so = ort.SessionOptions()
    so.intra_op_num_threads = THREADS
    so.inter_op_num_threads = 1
    return ort.InferenceSession(str(caminho), so, providers=['CPUExecutionProvider'])

SESS = {'onnx_fp32': sessao(FP32), 'onnx_int8': sessao(INT8)}
torch.set_num_threads(THREADS)

def _prob1(logits):
    logits = np.asarray(logits, dtype=np.float32)
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return (e / e.sum(axis=1, keepdims=True))[:, 1]

def _lotes(textos, batch):
    for i in range(0, len(textos), batch):
        yield [str(t) for t in textos[i:i + batch]]

def scores_torch(textos, batch=32):
    out = []
    for lote in _lotes(textos, batch):
        enc = tok(lote, truncation=True, max_length=MAXLEN, padding=True, return_tensors='pt')
        with torch.no_grad():
            out.append(_prob1(torch_model(**enc).logits.numpy()))
    return np.concatenate(out)

def scores_onnx(sess, textos, batch=32):
    out = []
    for lote in _lotes(textos, batch):
        enc = tok(lote, truncation=True, max_length=MAXLEN, padding=True, return_tensors='np')
        feed = {'input_ids': enc['input_ids'].astype(np.int64),
                'attention_mask': enc['attention_mask'].astype(np.int64)}
        out.append(_prob1(sess.run(['logits'], feed)[0]))
    return np.concatenate(out)

VARIANTES = {
    'torch_fp32': scores_torch,
    'onnx_fp32': lambda t, batch=32: scores_onnx(SESS['onnx_fp32'], t, batch),
    'onnx_int8': lambda t, batch=32: scores_onnx(SESS['onnx_int8'], t, batch),
}
print('teste rapido:', {k: round(float(f(['eu te odeio, seu lixo'])[0]), 4)
                        for k, f in VARIANTES.items()})

## 6 · Latência em CPU

Duas medidas, porque o produto usa as duas: **um texto por vez**, que é o site e
a extensão, e **lote de 32**, que é a análise de documento longo.

Referência do que roda hoje: 1,5 ms p50, um texto por vez.

In [ ]:
AMOSTRA = TE[TEXT_COL].astype(str).sample(80, random_state=42).tolist()

def mede(f, textos, batch, repeticoes=3):
    for t in textos[:5]:  # aquecimento: a primeira chamada monta cache e arena
        f([t], batch=1)
    tempos = []
    for _ in range(repeticoes):
        for i in range(0, len(textos), batch):
            lote = textos[i:i + batch]
            t0 = time.perf_counter()
            f(lote, batch=batch)
            tempos.append((time.perf_counter() - t0) * 1000 / len(lote))
    return np.percentile(tempos, 50), np.percentile(tempos, 95)

linhas = []
for nome, f in VARIANTES.items():
    p50_1, p95_1 = mede(f, AMOSTRA, batch=1)
    p50_32, _ = mede(f, AMOSTRA, batch=32)
    linhas.append({'variante': nome, 'tamanho_mb': TAM[nome], 'threads': THREADS,
                   'ms_p50_unitario': round(p50_1, 2), 'ms_p95_unitario': round(p95_1, 2),
                   'ms_p50_por_texto_lote32': round(p50_32, 2)})
    print(linhas[-1])

LATENCIA = pd.DataFrame(linhas)
print()
print(LATENCIA.to_string(index=False))
print('\nA CPU do Colab nao e a da Hetzner. Isto ordena as variantes;')
print('o numero final se mede na maquina.')

## 7 · Qualidade de cada variante

O limiar é reajustado **na validação para cada variante**, porque quantizar muda
a escala do score. Reaproveitar o limiar do fp32 no int8 seria comparar as duas
em pontos de operação diferentes.

In [ ]:
def best_threshold(y_true, y_score):
    y_true = np.asarray(y_true); y_score = np.asarray(y_score, dtype=float)
    cands = np.unique(np.quantile(y_score, np.linspace(0.02, 0.98, 97)))
    best_t, best_f = 0.5, -1.0
    for t in cands:
        f = f1_score(y_true, (y_score >= t).astype(int), average='macro', zero_division=0)
        if f > best_f: best_f, best_t = f, float(t)
    return best_t

SCORES, LIMIARES, qual = {}, {}, []
for nome, f in VARIANTES.items():
    s_va = f(VA[TEXT_COL].tolist())
    thr = best_threshold(VA['label'].values, s_va)
    s_te = f(TE[TEXT_COL].tolist())
    SCORES[nome], LIMIARES[nome] = s_te, thr
    for corte, mask in (('total', np.ones(len(TE), dtype=bool)),
                        ('en', (TE.language == 'en').values),
                        ('pt', (TE.language == 'pt').values)):
        y, sc_ = TE.loc[mask, 'label'].values, s_te[mask]
        yp = (sc_ >= thr).astype(int)
        qual.append({'variante': nome, 'limiar': round(thr, 4), 'corte': corte, 'n': int(len(y)),
                     'macro_f1': round(f1_score(y, yp, average='macro', zero_division=0), 4),
                     'recall_hate': round(recall_score(y, yp, zero_division=0), 4),
                     'precision_hate': round(precision_score(y, yp, zero_division=0), 4)})
    print(nome, 'ok')

QUALIDADE = pd.DataFrame(qual)
print()
print(QUALIDADE.to_string(index=False))

# quantos rotulos mudam de uma variante para a outra: o custo real da conversao
for a, b in (('onnx_fp32', 'torch_fp32'), ('onnx_int8', 'onnx_fp32')):
    ya = (SCORES[a] >= LIMIARES[a]).astype(int)
    yb = (SCORES[b] >= LIMIARES[b]).astype(int)
    print(f'{a} x {b}: concordancia {100*(ya == yb).mean():.2f}%'
          f' ({int((ya != yb).sum())} rotulos diferentes de {len(ya)})')

## 8 · O gate de viés de identidade

Vinte frases neutras ou positivas em português com termo de identidade. Nenhuma
é ódio, então toda marcação ali é falso positivo puro.

Referência a bater: o **stack servido hoje marca 3 de 20**. O modelo dedicado ao
PT marcava 9 e foi barrado; o limiar por idioma marcava 10 e foi barrado. **Se
uma variante passar de 3, ela não vai ao ar**, por melhor que seja o recall.

In [ ]:
from hsc.clean import clean_text
from hsc.config import data_config
PERFIL = data_config()['clean']['profiles']['light']

# o modelo foi treinado em text_clean: a sonda tem que passar pela mesma limpeza
neutras = [clean_text(f, PERFIL) for f in FRASES]
controles = [clean_text(f, PERFIL) for f in CONTROLES]

sonda = []
for nome, f in VARIANTES.items():
    thr = LIMIARES[nome]
    s_n, s_c = f(neutras), f(controles)
    fp, tp = int((s_n >= thr).sum()), int((s_c >= thr).sum())
    sonda.append({'variante': nome, 'limiar': round(thr, 4),
                  'fp_identidade': fp, 'n_sonda': len(FRASES),
                  'acertos_controle': tp, 'n_controle': len(CONTROLES),
                  'passa_no_gate': bool(fp <= 3)})
    print(f'{nome}: {fp}/{len(FRASES)} falsos positivos de identidade,'
          f' {tp}/{len(CONTROLES)} controles | {"PASSA" if fp <= 3 else "REPROVA"}')
    for frase, s in zip(FRASES, s_n):
        if s >= thr:
            print(f'    {s:.3f}  {frase}')

SONDA = pd.DataFrame(sonda)
print()
print(SONDA.to_string(index=False))

## 9 · Empacotar

As tabelas voltam por download. O ONNX int8 fica no Drive: é ele que sobe para a
Hetzner, se passar nos três testes.

In [ ]:
os.makedirs('/content/saida', exist_ok=True)
LATENCIA.to_csv('/content/saida/xlmr_latencia_cpu.csv', index=False)
QUALIDADE.to_csv('/content/saida/xlmr_qualidade_variantes.csv', index=False)
SONDA.to_csv('/content/saida/xlmr_sonda_identidade.csv', index=False)
json.dump({'limiares': {k: round(float(v), 4) for k, v in LIMIARES.items()},
           'tamanhos_mb': TAM, 'threads': THREADS, 'max_length': MAXLEN,
           'model_id': META['model_id']},
          open('/content/saida/xlmr_serve_meta.json', 'w'), indent=2)

ALVO = '/content/xlmr_bench_resultado.zip'
with zipfile.ZipFile(ALVO, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in Path('/content/saida').iterdir():
        z.write(f, f'reports/tables/{f.name}')

DEST_INT8 = Path(DRIVE_DIR) / 'modelos' / f"{META['model_id']}_onnx_int8"
DEST_INT8.mkdir(parents=True, exist_ok=True)
shutil.copy(INT8, DEST_INT8 / 'model_int8.onnx')
tok.save_pretrained(DEST_INT8)
shutil.copy('/content/saida/xlmr_serve_meta.json', DEST_INT8 / 'xlmr_serve_meta.json')
shutil.copy(ALVO, Path(DRIVE_DIR) / 'xlmr_bench_resultado.zip')
print('int8 e tokenizer no Drive:', DEST_INT8)

try:
    from google.colab import files
    files.download(ALVO)
except ImportError:
    print('rodando local:', ALVO)

---

**Me mande:** o `xlmr_bench_resultado.zip` e a saída das células 4, 6, 7 e 8.

Com isso eu decido três coisas de uma vez: se serve int8 ou fp32, qual limiar vai
para o registry, e se o modelo pode mesmo substituir o stack. Se a célula 8
reprovar todas as variantes, o twitter-XLM-R cai como o modelo dedicado caiu, e o
caminho passa a ser só o corpus sintético.